In [1]:
import os
from dotenv import load_dotenv

# This looks for the .env file and loads it into your environment
load_dotenv() 

# Now you can access them safely
gemini_key = os.getenv("GEMINI_API_KEY")
mongo_uri = os.getenv("MONGO_URI")

if not gemini_key or not mongo_uri:
    print("❌ Error: Keys not found! Check your .env file.")

In [2]:
from google import genai
from google.genai import types
import os

# 1. Initialize with explicit key to rule out environment issues
client = genai.Client(api_key=os.environ.get("GEMINI_API_KEY"))

# 2. Use the current stable model (text-embedding-004 is deprecated)
model_id = "gemini-embedding-001" 

def get_embedding(text, task_type="RETRIEVAL_DOCUMENT"):
    try:
        response = client.models.embed_content(
            model=model_id,
            contents=text,
            config=types.EmbedContentConfig(
                task_type=task_type
            )
        )
        # Access the first embedding result
        return response.embeddings[0].values
    except Exception as e:
        print(f"Error caught: {e}")
        return None

# Test call
vector = get_embedding("abdul kalam")
if vector:
    print(f"Success! Vector length: {len(vector)}")

Success! Vector length: 3072


In [3]:
import os
import glob
import pdfplumber
from rapidocr_onnxruntime import RapidOCR
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 1. Configuration
engine = RapidOCR()
directory_path = "../result_pdfs"
all_docs = []

# 2. Get list of all PDF files in the directory
pdf_files = glob.glob(os.path.join(directory_path, "*.pdf"))
print(f"📂 Found {len(pdf_files)} PDFs in {directory_path}...")

# 3. Loop through every file
for file_path in pdf_files:
    file_name = os.path.basename(file_path)
    print(f"\n📄 Processing: {file_name}")
    
    try:
        with pdfplumber.open(file_path) as pdf:
            for i, page in enumerate(pdf.pages):
                # Try normal text extraction
                text = page.extract_text()
                
                # If page is empty/scanned, force OCR
                if not text or len(text.strip()) < 10:
                    print(f"  ⚠️ Page {i+1} is scanned. Running OCR...")
                    img = page.to_image(resolution=300).original
                    result, _ = engine(img)
                    if result:
                        text = "\n".join([line[1] for line in result])
                
                if text:
                    # We store the file_name in metadata so the AI can cite its source!
                    all_docs.append(Document(
                        page_content=text, 
                        metadata={
                            "source": file_name, 
                            "page": i+1,
                            "path": file_path
                        }
                    ))
    except Exception as e:
        print(f"  ❌ Error reading {file_name}: {e}")

# 4. Final Chunking
if all_docs:
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap=100,
        separators=["\n\n", "\n", " ", ""]
    )
    documents = text_splitter.split_documents(all_docs)
    print(f"\n✅ Total Files Processed: {len(pdf_files)}")
    print(f"✅ Total Chunks Created: {len(documents)}")
else:
    print("❌ No text could be extracted from any files.")

📂 Found 1 PDFs in ../result_pdfs...

📄 Processing: B_Tech_Ordinance_2016-17.pdf

✅ Total Files Processed: 1
✅ Total Chunks Created: 43


In [4]:
texts = [doc.page_content for doc in documents]

print(f"✅ Ready to embed {len(texts)} chunks.") 

✅ Ready to embed 43 chunks.


In [5]:
import time
import re

# 1. Configuration for Free Tier
batch_size = 30  # Smaller batches to stay under TPM (Tokens Per Minute) limits
all_embeddings = []

print(f"🔄 Starting embedding for {len(texts)} chunks...")

# 2. Loop with error-aware retries
i = 0
while i < len(texts):
    mini_batch = texts[i : i + batch_size]
    print(f"📡 Processing chunks {i} to {min(i + batch_size, len(texts))}...")
    
    try:
        response = client.models.embed_content(
            model=model_id,
            contents=mini_batch,
            config=types.EmbedContentConfig(task_type="RETRIEVAL_DOCUMENT")
        )
        all_embeddings.extend([emb.values for emb in response.embeddings])
        i += batch_size # Move to next batch only on success
        
        # ☕ Mandatory 12-second pause to stay under ~5 RPM
        if i < len(texts):
            print("☕ Cooling down for 12 seconds to respect Free Tier limits...")
            time.sleep(12)
            
    except Exception as e:
        error_msg = str(e)
        if "429" in error_msg:
            # Look for "Please retry in X.Xs" in the error message
            wait_match = re.search(r"retry in (\d+\.?\d*)s", error_msg)
            wait_time = float(wait_match.group(1)) + 2 if wait_match else 40
            
            print(f"⚠️ Rate limit hit! API says wait {wait_time}s. Sleeping...")
            time.sleep(wait_time)
        else:
            print(f"❌ Unexpected Error: {e}")
            break

# 3. Final Verification
if len(all_embeddings) == len(texts):
    docs_to_insert = [
        {
            "text": texts[j],
            "embedding": all_embeddings[j],
            "metadata": documents[j].metadata 
        } 
        for j in range(len(all_embeddings))
    ]
    print(f"\n✅ Success! All {len(docs_to_insert)} chunks embedded.")

🔄 Starting embedding for 43 chunks...
📡 Processing chunks 0 to 30...
☕ Cooling down for 12 seconds to respect Free Tier limits...
📡 Processing chunks 30 to 43...

✅ Success! All 43 chunks embedded.


In [6]:
docs_to_insert

[{'text': 'DR. A.P.J. ABDUL KALAM TECHNICAL UNIVERSITY\nLUCKNOW\nRules and Regulations\nFor\nUndergraduate Course\n(B. Tech. /B. Pharmacy/BHMCT/BFAD)\nOn\nChoice Based Credit System\n(Effective from the Session: 2016-17)',
  'embedding': [0.025926948,
   0.006113531,
   0.003490349,
   -0.04959578,
   0.0055696475,
   0.013334468,
   0.012044625,
   -0.012682621,
   -0.0007704877,
   0.010082076,
   -0.022290848,
   -0.011227598,
   0.0062573045,
   0.030935777,
   0.09096162,
   -0.0023183026,
   -0.012962746,
   -0.016011532,
   -0.014496904,
   -0.016242016,
   -0.0031712244,
   0.008119194,
   -0.0064512,
   -0.0073254514,
   -0.0026599518,
   -0.0028736107,
   0.015155463,
   0.0051527703,
   0.037300486,
   0.031627834,
   -0.0024314714,
   0.0060248375,
   0.014703388,
   0.015427179,
   0.016167555,
   0.020165725,
   0.008934045,
   -0.005433222,
   0.003891157,
   0.0051405975,
   -0.008904176,
   0.044283614,
   0.011429762,
   -0.005587567,
   -0.016657664,
   -0.002648643,

In [7]:
from pymongo import MongoClient
import os

# 1. Your connection string
MONGO_URI = os.getenv("MONGO_URI")
client = MongoClient(MONGO_URI)

# 2. Define your actual project database and collection
# (This creates them automatically upon the first insertion)
db = client["rag_pdfs"] 
collection = db["result"]

# 3. Perform the insertion
if docs_to_insert:
    result = collection.insert_many(docs_to_insert)
    print(f"✅ Success! Created database 'RAG_db' and inserted {len(result.inserted_ids)} documents.")
else:
    print("❌ Error: docs_to_insert is empty.")

✅ Success! Created database 'RAG_db' and inserted 43 documents.


In [8]:
result

InsertManyResult([ObjectId('69ad13d7ce97ef8ca4c0f021'), ObjectId('69ad13d7ce97ef8ca4c0f022'), ObjectId('69ad13d7ce97ef8ca4c0f023'), ObjectId('69ad13d7ce97ef8ca4c0f024'), ObjectId('69ad13d7ce97ef8ca4c0f025'), ObjectId('69ad13d7ce97ef8ca4c0f026'), ObjectId('69ad13d7ce97ef8ca4c0f027'), ObjectId('69ad13d7ce97ef8ca4c0f028'), ObjectId('69ad13d7ce97ef8ca4c0f029'), ObjectId('69ad13d7ce97ef8ca4c0f02a'), ObjectId('69ad13d7ce97ef8ca4c0f02b'), ObjectId('69ad13d7ce97ef8ca4c0f02c'), ObjectId('69ad13d7ce97ef8ca4c0f02d'), ObjectId('69ad13d7ce97ef8ca4c0f02e'), ObjectId('69ad13d7ce97ef8ca4c0f02f'), ObjectId('69ad13d7ce97ef8ca4c0f030'), ObjectId('69ad13d7ce97ef8ca4c0f031'), ObjectId('69ad13d7ce97ef8ca4c0f032'), ObjectId('69ad13d7ce97ef8ca4c0f033'), ObjectId('69ad13d7ce97ef8ca4c0f034'), ObjectId('69ad13d7ce97ef8ca4c0f035'), ObjectId('69ad13d7ce97ef8ca4c0f036'), ObjectId('69ad13d7ce97ef8ca4c0f037'), ObjectId('69ad13d7ce97ef8ca4c0f038'), ObjectId('69ad13d7ce97ef8ca4c0f039'), ObjectId('69ad13d7ce97ef8ca4c0f0

In [9]:
from pymongo.operations import SearchIndexModel
import time

# 1. Define the Index Name
index_name = "vector_index"

# 2. Create the Search Index Model
# Ensure numDimensions is 3072 for gemini-embedding-001
search_index_model = SearchIndexModel(
    definition={
        "fields": [
            {
                "type": "vector",
                "numDimensions": 3072, # Correct for gemini-embedding-001
                "path": "embedding",
                "similarity": "cosine"
            }
        ]
    },
    name=index_name,
    type="vectorSearch"
)

# 3. Create the index on your collection
print(f"Creating index '{index_name}'...")
collection.create_search_index(model=search_index_model)

Creating index 'vector_index'...


'vector_index'

In [10]:
print("Polling to check if the index is ready. This may take up to a minute.")
predicate=None
if predicate is None:
   predicate = lambda index: index.get("queryable") is True

while True:
   indices = list(collection.list_search_indexes(index_name))
   if len(indices) and predicate(indices[0]):
      break
   time.sleep(5)
print(index_name + " is ready for querying.")

Polling to check if the index is ready. This may take up to a minute.
vector_index is ready for querying.


In [16]:
import requests
from google import genai
from google.genai import types

def get_query_results(query, roll_no):
    """Fetches student marks via API and uses RAG for result analysis."""
    client = genai.Client()
    
    # 1. Fetch Student Result using your specific URL
    api_url = f"https://singularity-server.devxoshakya.workers.dev/api/result/by-rollno?rollNo={roll_no}"
    try:
        response = requests.get(api_url)
        api_data = response.json()
        student_context = api_data.get("data", "No data found") if api_data.get("success") else "API error"
    except Exception as e:
        student_context = f"Network error: {str(e)}"

    # 2. Generate Embedding using the updated 2026 model ID
    # Use 'gemini-embedding-001' for the modern google-genai SDK
    model_id = "gemini-embedding-001"
    embed_response = client.models.embed_content(
        model=model_id,
        contents=query,
        config=types.EmbedContentConfig(task_type="RETRIEVAL_QUERY")
    )
    query_embedding = embed_response.embeddings[0].values

    # 3. MongoDB Vector Search Pipeline
    pipeline = [
        {
            "$vectorSearch": {
                "index": "vector_index",      # Ensure this index exists on your new collection
                "queryVector": query_embedding,
                "path": "embedding",
                "numCandidates": 100,
                "limit": 5
            }
        },
        {"$project": {"_id": 0, "text": 1}}
    ]

    # Execute search on your grading_rules collection
    rules_results = list(collection.aggregate(pipeline))
    rules_context = "\n".join([doc['text'] for doc in rules_results])

    # 4. Final Analysis Prompt
    prompt = f"""
    Context: {rules_context}
    Student Data: {student_context}
    Query: {query}

    Instructions: Analyze the student's result against the rules precisely. 
    If the student asks about 'PWG' (Pass With Grace), check the grace marks policy.
    """

    # Generate response
    result = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt,
        config=types.GenerateContentConfig(temperature=0)
    )

    return result.text

In [19]:
get_query_results("how can i pass how much more credits do i need tell me in detail?", 2400680109003)

'Based on the provided rules and your academic data, here\'s a detailed explanation of how you can pass and the credits you need to earn:\n\n**Your Current Status:**\n*   **Year:** 2nd Year\n*   **Semester 3 SGPA:** 6.96 (Passed)\n*   **Semester 4 SGPA:** 4.35 (Failed, as it\'s below the minimum required 5.0)\n*   **Carry Over Papers (COP):** BAS403, BCS401 (These are the subjects you need to clear).\n*   **Latest Result Status:** PCP (Partially Cleared Papers, indicating you have carry-overs).\n\n**Key Rules for Passing and Promotion:**\n\n1.  **Individual Subject Pass (Rule 9.1):**\n    *   **Theory Subjects (9.1a):** You must secure a minimum of **30% of the maximum marks in the University Examination (External)** AND **40% of the aggregate marks (Internal + External)**. The minimum passing grade is "E".\n    *   **Practical Subjects (9.1b):** You must secure a minimum of **50% of the maximum marks in the University Examination (External)** AND **40% of the aggregate marks (Internal